# FinSphere AI — Enterprise Credit Risk & Delinquency Prediction Module
### Predictive Underwriting, Adverse Action Explainability & Regulatory Compliance using XGBoost & TreeSHAP

**Dataset:** Give Me Some Credit (`cs-training.csv`)  
**Target:** `SeriousDlqin2yrs` (1 = Borrower experienced 90+ days delinquency within 2 years, 0 = Did not)  
**Author:** AI/ML Engineering Team  

---
## Business Problem & ML Architecture
Credit risk models evaluate the probability of borrower default. Unlike black-box regimes, credit underwriting models are subject to strict regulatory compliance (e.g. Equal Credit Opportunity Act - ECOA, Fair Credit Reporting Act - FCRA), requiring **transparent, adverse-action factor explanations**.

This module implements:
1. **Financial Feature Engineering**: Capping extreme utilization/debt outliers, computing non-linear debt burden stress metrics, and building delinquency severity weighted scores.
2. **Cost-Sensitive Gradient Boosting**: XGBoost with `scale_pos_weight` tuned for credit risk portfolios.
3. **Adverse Action Explanations via TreeSHAP**: Providing exact quantitative reasons for risk classification.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, average_precision_score, precision_recall_curve, f1_score
)

# Visual styling
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'


## 1. Load Data & Financial Data Cleaning

In [ ]:
risk_path = 'API-layer/cs-training.csv' if os.path.exists('API-layer/cs-training.csv') else 'cs-training.csv'
risk = pd.read_csv(risk_path)
if 'Unnamed: 0' in risk.columns:
    risk = risk.drop(columns=['Unnamed: 0'])

print(f"Dataset Shape: {risk.shape}")
print(f"Missing Values:\n{risk.isnull().sum()[risk.isnull().sum() > 0]}")
risk.head(3)


## 2. Exploratory Data Analysis & Financial Risk Signals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

counts = risk['SeriousDlqin2yrs'].value_counts()
axes[0].bar(['Performing Loans (0)', 'Delinquent (1)'], counts.values, color=['#3B82F6', '#EF4444'])
axes[0].set_title('Loan Portfolio Status', fontweight='bold')
axes[0].set_ylabel('Applicant Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 2000, f"{v:,}\n({v/counts.sum()*100:.2f}%)", ha='center', fontweight='semibold')

sns.histplot(risk['age'], bins=40, ax=axes[1], color='#3B82F6', kde=True)
axes[1].set_title('Borrower Age Distribution', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Overall Portfolio Delinquency Rate: {risk['SeriousDlqin2yrs'].mean()*100:.2f}%")


## 3. Financial Feature Engineering & Non-Linear Stress Indices

In [ ]:
LATE_COLS = [
    'NumberOfTime30-59DaysPastDueNotWorse',
    'NumberOfTimes90DaysLate',
    'NumberOfTime60-89DaysPastDueNotWorse'
]

def clean_and_engineer_risk(df, quantiles=None, is_train=False):
    df = df.copy()
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    df = df[df['age'] > 0] if 'SeriousDlqin2yrs' in df.columns else df
    df['age'] = df['age'].clip(lower=18, upper=100)
    df['NumberOfDependents'] = df['NumberOfDependents'].fillna(0).clip(0, 20)
    
    if is_train:
        income_median = float(df['MonthlyIncome'].median())
        util_q99 = float(df['RevolvingUtilizationOfUnsecuredLines'].quantile(0.99))
        debt_q99 = float(df['DebtRatio'].quantile(0.99))
        quantiles = {'income_median': income_median, 'util_q99': util_q99, 'debt_q99': debt_q99}
    
    income_med = quantiles.get('income_median', 5400.0) if quantiles else 5400.0
    util_cap = quantiles.get('util_q99', 2.0) if quantiles else 2.0
    debt_cap = quantiles.get('debt_q99', 5.0) if quantiles else 5.0

    df['MonthlyIncome'] = df['MonthlyIncome'].fillna(income_med).clip(lower=0)
    for col in LATE_COLS:
        df[col] = df[col].clip(lower=0, upper=18)
        
    df['RevolvingUtilizationOfUnsecuredLines'] = df['RevolvingUtilizationOfUnsecuredLines'].clip(lower=0, upper=util_cap)
    df['DebtRatio'] = df['DebtRatio'].clip(lower=0, upper=debt_cap)
    
    # Financial Stress Features
    df['total_past_due'] = df[LATE_COLS].sum(axis=1)
    df['weighted_delinquency_score'] = (
        1.0 * df['NumberOfTime30-59DaysPastDueNotWorse'] +
        2.0 * df['NumberOfTime60-89DaysPastDueNotWorse'] +
        3.5 * df['NumberOfTimes90DaysLate']
    )
    df['income_per_dependent'] = df['MonthlyIncome'] / (df['NumberOfDependents'] + 1.0)
    df['disposable_income_estimate'] = df['MonthlyIncome'] * (1.0 - np.clip(df['DebtRatio'], 0, 0.95))
    df['utilization_stress_index'] = (
        (df['RevolvingUtilizationOfUnsecuredLines'] > 0.8).astype(int) +
        (df['RevolvingUtilizationOfUnsecuredLines'] > 1.0).astype(int)
    )
    df['real_estate_to_open_lines_ratio'] = df['NumberRealEstateLoansOrLines'] / (df['NumberOfOpenCreditLinesAndLoans'] + 1.0)
    
    return df, quantiles

risk_cleaned, risk_q = clean_and_engineer_risk(risk, is_train=True)
print("Engineered risk features sample:")
risk_cleaned[['weighted_delinquency_score', 'utilization_stress_index', 'income_per_dependent']].head()


## 4. Train/Test Stratified Split & Preprocessing

In [ ]:
feature_cols = [
    'RevolvingUtilizationOfUnsecuredLines', 'age',
    'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio',
    'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
    'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines',
    'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents',
    'total_past_due', 'weighted_delinquency_score',
    'income_per_dependent', 'disposable_income_estimate',
    'utilization_stress_index', 'real_estate_to_open_lines_ratio'
]

y = risk_cleaned['SeriousDlqin2yrs'].values
X = risk_cleaned[feature_cols].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")


## 5. Model Training: Random Forest vs. Optimized XGBoost

In [ ]:
# 1. Random Forest Baseline
rf_risk = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced_subsample', n_jobs=-1, random_state=42)
rf_risk.fit(X_train, y_train)
rf_prob = rf_risk.predict_proba(X_test)[:, 1]

# 2. Optimized XGBoost
neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)
scale_weight = float(neg_count / max(pos_count, 1))

xgb_risk = XGBClassifier(
    n_estimators=250,
    max_depth=5,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    scale_pos_weight=min(scale_weight, 10.0),
    eval_metric='aucpr',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
xgb_risk.fit(X_train, y_train)
xgb_prob = xgb_risk.predict_proba(X_test)[:, 1]

print("Credit Risk Models Trained.")


## 6. Model Evaluation: ROC & Precision-Recall Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for proba, name, color in [(rf_prob, 'Random Forest Baseline', '#64748B'), (xgb_prob, 'Optimized XGBoost (Proposed)', '#3B82F6')]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {auc:.4f})", color=color, linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_title('ROC Curve (Credit Delinquency)', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend()

for proba, name, color in [(rf_prob, 'Random Forest Baseline', '#64748B'), (xgb_prob, 'Optimized XGBoost (Proposed)', '#3B82F6')]:
    prec, rec, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC = {ap:.4f})", color=color, linewidth=2)
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Explainable AI & Adverse Action Notice Generation (TreeSHAP)

In [ ]:
sample_X_r = X_test.sample(n=500, random_state=42)
explainer_r = shap.TreeExplainer(xgb_risk)
shap_vals_r = explainer_r.shap_values(sample_X_r)

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_vals_r, sample_X_r, plot_type="bar", show=False)
plt.title("Credit Risk Factor Drivers (TreeSHAP Mean Importance)", fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Summary & Regulatory Applications
1. **Underwriting Precision**: XGBoost with financial stress index features delivers higher ROC-AUC and PR-AUC than baseline random forests.
2. **Adverse Action Notices**: SHAP values directly identify the top contributing factors when an application is classified as High Risk (e.g. revolving utilization >80%, past-due history).
3. **Integration**: Serialized into `models/risk_model.pkl` for serving via `/predict/risk` and `/explain/risk`.
